In [0]:
try:
    spark.conf.set(
    "fs.azure.account.key.accstoragedeproject.dfs.core.windows.net",
    dbutils.secrets.get(scope="key-vault-scope", key="storage-key")
    )
except:
    raise Exception("Nie można uzyskać dostępu do Key Vault.")

In [0]:
import pyspark.sql.functions as F

In [0]:
storage_name = 'accstoragedeproject'

silver_path = f"abfss://silver@{storage_name}.dfs.core.windows.net/cleaned_joined_orders"
gold_path = f'abfss://gold@{storage_name}.dfs.core.windows.net/'
df_silver = spark.read.format("delta").load(silver_path)

In [0]:
df_gold_sales_performance = df_silver.groupBy(
    'Category',
    'City',
    'State',
    F.date_format(F.col("Order_Date"), 'yyyy-MM').alias('Month_Year')
).agg(
    F.sum('Amount').alias('Total_Sales'),
    F.sum('Profit').alias('Total_Profit'),
    F.count("Order_ID").alias('Number_of_Orders'),
    F.sum("Quantity").alias('Total_Items_sold')
)

In [0]:
df_gold_sales_performance.write.format('delta').mode('overwrite').save(gold_path + 'sales_performance')